In [ ]:
import json
import logging
import uuid
from datetime import datetime

from delta.tables import DeltaTable
from pyspark.sql import Row
from pyspark.sql.functions import (
    current_timestamp, lit, col, sha2, concat_ws, coalesce
)
from pyspark.sql import functions as F

In [ ]:


bronze_dfs = {}


bronze_dfs["d_labitems"] = spark.read.table(f"{bronze_full_schema}.raw_d_labitems")
bronze_dfs["labevents"] = spark.read.table(f"{bronze_full_schema}.raw_labevents")
bronze_dfs["microbiologyevents"] = spark.read.table(f"{bronze_full_schema}.raw_microbiologyevents")
bronze_dfs["prescriptions"] = spark.read.table(f"{bronze_full_schema}.raw_prescriptions")


In [ ]:
#lab Items


silver_dfs = {}

d_labitems_df = bronze_dfs["d_labitems"]

canonical_label_map = F.create_map(
    F.lit("c diff toxin antigen assay"),
    F.lit("C diff Toxin Antigen Assay"),

    F.lit("leuks"),
    F.lit("LEUKS"),

    F.lit("mesothelial cells"),
    F.lit("Mesothelial Cells"),

    F.lit("rbc"),
    F.lit("RBC")
)

d_labitems_clean = (
    d_labitems_df

    # Convert itemid to StringType and remove surrounding whitespace
    .withColumn(
        "itemid",
        F.trim(F.col("itemid").cast("string"))
    )

    # Trim label and remove repeated internal whitespace
    .withColumn(
        "label",
        F.regexp_replace(
            F.trim(F.col("label")),
            r"\s+",
            " "
        )
    )

    # Convert empty labels to null
    .withColumn(
        "label",
        F.when(
            F.col("label") == "",
            F.lit(None)
        ).otherwise(F.col("label"))
    )

    # Correct known inconsistent casing
    # Other labels remain unchanged, preserving clinical abbreviations
    .withColumn(
        "label",
        F.coalesce(
            canonical_label_map[F.lower(F.col("label"))],
            F.col("label")
        )
    )

    # Replace missing labels with "unavailable"
    .withColumn(
        "label",
        F.coalesce(
            F.col("label"),
            F.lit("unavailable")
        )
    )

    # Trim fluid and convert empty values to null
    .withColumn(
        "fluid",
        F.when(
            F.trim(F.col("fluid")) == "",
            F.lit(None)
        ).otherwise(F.trim(F.col("fluid")))
    )

    # Trim category and convert empty values to null
    .withColumn(
        "category",
        F.when(
            F.trim(F.col("category")) == "",
            F.lit(None)
        ).otherwise(F.trim(F.col("category")))
    )

    # Remove duplicate item IDs
    .dropDuplicates(["itemid"])
)

silver_dfs["d_labitems"] = d_labitems_clean

In [ ]:
#lab events


labevents_df = bronze_dfs["labevents"]

labevents_clean = (
    labevents_df

    # Convert numeric identifiers to clean StringType values
    # Casting through long prevents values such as "29600294.0"
    .withColumn(
        "labevent_id",
        F.col("labevent_id").cast("long").cast("string")
    )
    .withColumn(
        "subject_id",
        F.col("subject_id").cast("long").cast("string")
    )
    .withColumn(
        "hadm_id",
        F.col("hadm_id").cast("long").cast("string")
    )
    .withColumn(
        "specimen_id",
        F.col("specimen_id").cast("long").cast("string")
    )
    .withColumn(
        "itemid",
        F.col("itemid").cast("long").cast("string")
    )

    # order_provider_id is alphanumeric, so it should not be cast through long
    .withColumn(
        "order_provider_id",
        F.when(
            F.trim(F.col("order_provider_id").cast("string")) == "",
            F.lit(None)
        ).otherwise(
            F.trim(F.col("order_provider_id").cast("string"))
        )
    )

    # Convert date-time columns to TimestampType
    .withColumn(
        "charttime",
        F.to_timestamp(F.col("charttime"))
    )
    .withColumn(
        "storetime",
        F.to_timestamp(F.col("storetime"))
    )

    # Preserve value as text, trim it and normalise repeated whitespace
    .withColumn(
        "value",
        F.regexp_replace(
            F.trim(F.col("value").cast("string")),
            r"\s+",
            " "
        )
    )
    .withColumn(
        "value",
        F.when(
            F.col("value") == "",
            F.lit(None)
        ).otherwise(F.col("value"))
    )

    # Convert the numeric result to DoubleType
    .withColumn(
        "valuenum",
        F.col("valuenum").cast("double")
    )

    # Trim units and convert blank units to null
    .withColumn(
        "valueuom",
        F.when(
            F.trim(F.col("valueuom").cast("string")) == "",
            F.lit(None)
        ).otherwise(
            F.trim(F.col("valueuom").cast("string"))
        )
    )

    # Convert reference ranges to DoubleType
    .withColumn(
        "ref_range_lower",
        F.col("ref_range_lower").cast("double")
    )
    .withColumn(
        "ref_range_upper",
        F.col("ref_range_upper").cast("double")
    )

    # Standardise the laboratory flag
    .withColumn(
        "flag",
        F.when(
            F.trim(F.col("flag").cast("string")) == "",
            F.lit(None)
        ).otherwise(
            F.lower(F.trim(F.col("flag").cast("string")))
        )
    )

    # Standardise priority
    .withColumn(
        "priority",
        F.when(
            F.trim(F.col("priority").cast("string")) == "",
            F.lit(None)
        ).otherwise(
            F.upper(F.trim(F.col("priority").cast("string")))
        )
    )

    # Trim comments while preserving genuine null values
    .withColumn(
        "comments",
        F.when(
            F.trim(F.col("comments").cast("string")) == "",
            F.lit(None)
        ).otherwise(
            F.trim(F.col("comments").cast("string"))
        )
    )
)

# Bring in the cleaned laboratory-item label temporarily
# This is required to identify Voided Specimen results correctly
labevents_clean = (
    labevents_clean
    .join(
        F.broadcast(
            silver_dfs["d_labitems"]
            .select(
                "itemid",
                F.col("label").alias("_labitem_label")
            )
        ),
        on="itemid",
        how="left"
    )

    # Classify each laboratory result
    .withColumn(
        "result_type",
        F.when(
            F.lower(F.trim(F.col("_labitem_label"))) == "voided specimen",
            F.lit("voided")
        )
        .when(
            F.col("valuenum").isNotNull(),
            F.lit("numeric")
        )
        .when(
            F.col("value").isNotNull(),
            F.lit("qualitative")
        )
        .otherwise(
            F.lit("missing")
        )
    )

    # Remove the temporary lookup column
    .drop("_labitem_label")

    # Remove only completely identical rows
    .dropDuplicates()
)

silver_dfs["labevents"] = labevents_clean

In [ ]:
#prescriptions

# Load the Bronze prescriptions DataFrame
prescriptions_df = bronze_dfs["prescriptions"]

# Define patterns for single values and ranges
numeric_dose_pattern = r"^[0-9]*\.?[0-9]+$"
range_dose_pattern = r"^[0-9]*\.?[0-9]+-[0-9]*\.?[0-9]+$"

prescriptions_clean = (
    prescriptions_df

    # Convert subject ID to string
    .withColumn(
        "subject_id",
        F.col("subject_id").cast("long").cast("string")
    )

    # Convert admission ID to string
    .withColumn(
        "hadm_id",
        F.col("hadm_id").cast("long").cast("string")
    )

    # Convert pharmacy ID to string
    .withColumn(
        "pharmacy_id",
        F.col("pharmacy_id").cast("long").cast("string")
    )

    # Convert POE sequence to string
    .withColumn(
        "poe_seq",
        F.col("poe_seq").cast("long").cast("string")
    )

    # Trim POE ID and convert blanks to null
    .withColumn(
        "poe_id",
        F.when(
            F.trim(F.col("poe_id").cast("string")) == "",
            F.lit(None)
        ).otherwise(
            F.trim(F.col("poe_id").cast("string"))
        )
    )

    # Trim provider ID and convert blanks to null
    .withColumn(
        "order_provider_id",
        F.when(
            F.trim(F.col("order_provider_id").cast("string")) == "",
            F.lit(None)
        ).otherwise(
            F.trim(F.col("order_provider_id").cast("string"))
        )
    )

    # Convert start time to timestamp
    .withColumn(
        "starttime",
        F.to_timestamp(F.col("starttime"))
    )

    # Convert stop time to timestamp
    .withColumn(
        "stoptime",
        F.to_timestamp(F.col("stoptime"))
    )

    # Trim and uppercase drug type
    .withColumn(
        "drug_type",
        F.when(
            F.trim(F.col("drug_type")) == "",
            F.lit(None)
        ).otherwise(
            F.upper(F.trim(F.col("drug_type")))
        )
    )

    # Preserve the original drug name
    .withColumn(
        "drug_raw",
        F.col("drug")
    )

    # Trim drug name and remove repeated spaces
    .withColumn(
        "drug",
        F.when(
            F.trim(F.col("drug")) == "",
            F.lit(None)
        ).otherwise(
            F.regexp_replace(
                F.trim(F.col("drug")),
                r"\s+",
                " "
            )
        )
    )

    # Convert drug name to title case
    .withColumn(
        "drug",
        F.when(
            F.col("drug").isNull(),
            F.lit(None)
        ).otherwise(
            F.initcap(F.lower(F.col("drug")))
        )
    )

    # Trim and uppercase formulary code
    .withColumn(
        "formulary_drug_cd",
        F.when(
            F.trim(F.col("formulary_drug_cd")) == "",
            F.lit(None)
        ).otherwise(
            F.upper(F.trim(F.col("formulary_drug_cd")))
        )
    )

    # Trim GSN and preserve leading zeros
    .withColumn(
        "gsn",
        F.when(
            F.trim(F.col("gsn").cast("string")) == "",
            F.lit(None)
        ).otherwise(
            F.trim(F.col("gsn").cast("string"))
        )
    )

    # Split multiple GSN codes into an array
    .withColumn(
        "gsn_codes",
        F.when(
            F.col("gsn").isNotNull(),
            F.split(F.col("gsn"), r"\s+")
        ).otherwise(
            F.lit(None).cast("array<string>")
        )
    )

    # Convert NDC to string and remove decimal suffix
    .withColumn(
        "ndc",
        F.col("ndc").cast("long").cast("string")
    )

    # Convert zero NDC to null
    .withColumn(
        "ndc",
        F.when(
            F.col("ndc") == "0",
            F.lit(None)
        ).otherwise(
            F.col("ndc")
        )
    )

    # Flag missing NDC values
    .withColumn(
        "missing_ndc_flag",
        F.col("ndc").isNull()
    )

    # Trim product strength and remove repeated spaces
    .withColumn(
        "prod_strength",
        F.when(
            F.trim(F.col("prod_strength")) == "",
            F.lit(None)
        ).otherwise(
            F.regexp_replace(
                F.trim(F.col("prod_strength")),
                r"\s+",
                " "
            )
        )
    )

    # Standardise mL unit casing
    .withColumn(
        "prod_strength",
        F.regexp_replace(
            F.col("prod_strength"),
            r"(?i)\bml\b",
            "mL"
        )
    )

    # Preserve the original prescribed dose
    .withColumn(
        "dose_val_rx_raw",
        F.col("dose_val_rx")
    )

    # Preserve the original dispensed-form value
    .withColumn(
        "form_val_disp_raw",
        F.col("form_val_disp")
    )

    # Trim dose and remove thousands separators
    .withColumn(
        "_dose_clean",
        F.regexp_replace(
            F.trim(F.col("dose_val_rx").cast("string")),
            ",",
            ""
        )
    )

    # Trim dispensed value and remove separators
    .withColumn(
        "_form_disp_clean",
        F.regexp_replace(
            F.trim(F.col("form_val_disp").cast("string")),
            ",",
            ""
        )
    )

    # Convert blank dose values to null
    .withColumn(
        "_dose_clean",
        F.when(
            F.col("_dose_clean") == "",
            F.lit(None)
        ).otherwise(
            F.col("_dose_clean")
        )
    )

    # Convert blank dispensed values to null
    .withColumn(
        "_form_disp_clean",
        F.when(
            F.col("_form_disp_clean") == "",
            F.lit(None)
        ).otherwise(
            F.col("_form_disp_clean")
        )
    )

    # Store the cleaned prescribed dose
    .withColumn(
        "dose_val_rx",
        F.col("_dose_clean")
    )

    # Store the cleaned dispensed-form value
    .withColumn(
        "form_val_disp",
        F.col("_form_disp_clean")
    )

    # Extract the minimum prescribed dose
    .withColumn(
        "dose_min",
        F.when(
            F.col("_dose_clean").rlike(numeric_dose_pattern)
            | F.col("_dose_clean").rlike(range_dose_pattern),
            F.regexp_extract(
                F.col("_dose_clean"),
                r"^([0-9]*\.?[0-9]+)",
                1
            ).cast("double")
        ).otherwise(
            F.lit(None).cast("double")
        )
    )

    # Extract the maximum prescribed dose
    .withColumn(
        "dose_max",
        F.when(
            F.col("_dose_clean").rlike(range_dose_pattern),
            F.regexp_extract(
                F.col("_dose_clean"),
                r"-([0-9]*\.?[0-9]+)$",
                1
            ).cast("double")
        )
        .when(
            F.col("_dose_clean").rlike(numeric_dose_pattern),
            F.col("dose_min")
        )
        .otherwise(
            F.lit(None).cast("double")
        )
    )

    # Extract the minimum dispensed value
    .withColumn(
        "form_disp_min",
        F.when(
            F.col("_form_disp_clean").rlike(numeric_dose_pattern)
            | F.col("_form_disp_clean").rlike(range_dose_pattern),
            F.regexp_extract(
                F.col("_form_disp_clean"),
                r"^([0-9]*\.?[0-9]+)",
                1
            ).cast("double")
        ).otherwise(
            F.lit(None).cast("double")
        )
    )

    # Extract the maximum dispensed value
    .withColumn(
        "form_disp_max",
        F.when(
            F.col("_form_disp_clean").rlike(range_dose_pattern),
            F.regexp_extract(
                F.col("_form_disp_clean"),
                r"-([0-9]*\.?[0-9]+)$",
                1
            ).cast("double")
        )
        .when(
            F.col("_form_disp_clean").rlike(numeric_dose_pattern),
            F.col("form_disp_min")
        )
        .otherwise(
            F.lit(None).cast("double")
        )
    )

    # Trim the prescribed-dose unit
    .withColumn(
        "dose_unit_rx",
        F.when(
            F.trim(F.col("dose_unit_rx")) == "",
            F.lit(None)
        ).otherwise(
            F.trim(F.col("dose_unit_rx"))
        )
    )

    # Trim the dispensed-form unit
    .withColumn(
        "form_unit_disp",
        F.when(
            F.trim(F.col("form_unit_disp")) == "",
            F.lit(None)
        ).otherwise(
            F.trim(F.col("form_unit_disp"))
        )
    )

    # Convert doses per day to integer
    .withColumn(
        "doses_per_24_hrs",
        F.col("doses_per_24_hrs").cast("integer")
    )

    # Trim and uppercase administration route
    .withColumn(
        "route",
        F.when(
            F.trim(F.col("route")) == "",
            F.lit(None)
        ).otherwise(
            F.upper(F.trim(F.col("route")))
        )
    )

    # Remove temporary parsing columns
    .drop(
        "_dose_clean",
        "_form_disp_clean"
    )

    # Remove completely identical records
    .dropDuplicates()
)

# Store the cleaned prescriptions DataFrame
silver_dfs["prescriptions"] = prescriptions_clean

In [ ]:


# Load the Bronze microbiology table
microbiology_df = bronze_dfs["microbiologyevents"]

microbiology_clean = (
    microbiology_df

    # Convert microbiology event ID to string
    .withColumn(
        "microevent_id",
        F.col("microevent_id").cast("long").cast("string")
    )

    # Convert patient ID to string
    .withColumn(
        "subject_id",
        F.col("subject_id").cast("long").cast("string")
    )

    # Convert admission ID to string and preserve nulls
    .withColumn(
        "hadm_id",
        F.col("hadm_id").cast("long").cast("string")
    )

    # Convert specimen ID to string
    .withColumn(
        "micro_specimen_id",
        F.col("micro_specimen_id").cast("long").cast("string")
    )

    # Trim provider ID and preserve nulls
    .withColumn(
        "order_provider_id",
        F.when(
            F.trim(F.col("order_provider_id").cast("string")) == "",
            F.lit(None)
        ).otherwise(
            F.trim(F.col("order_provider_id").cast("string"))
        )
    )

    # Convert chart date to DateType
    .withColumn(
        "chartdate",
        F.to_date(F.col("chartdate"))
    )

    # Convert chart time to TimestampType
    .withColumn(
        "charttime",
        F.to_timestamp(F.col("charttime"))
    )

    # Convert store date to DateType
    .withColumn(
        "storedate",
        F.to_date(F.col("storedate"))
    )

    # Convert store time to TimestampType
    .withColumn(
        "storetime",
        F.to_timestamp(F.col("storetime"))
    )

    # Convert specimen-item ID to string
    .withColumn(
        "spec_itemid",
        F.col("spec_itemid").cast("long").cast("string")
    )

    # Trim specimen description
    .withColumn(
        "spec_type_desc",
        F.when(
            F.trim(F.col("spec_type_desc")) == "",
            F.lit(None)
        ).otherwise(
            F.trim(F.col("spec_type_desc"))
        )
    )

    # Convert test-item ID to string
    .withColumn(
        "test_itemid",
        F.col("test_itemid").cast("long").cast("string")
    )

    # Trim test name
    .withColumn(
        "test_name",
        F.when(
            F.trim(F.col("test_name")) == "",
            F.lit(None)
        ).otherwise(
            F.trim(F.col("test_name"))
        )
    )

    # Convert organism ID to string and preserve nulls
    .withColumn(
        "org_itemid",
        F.col("org_itemid").cast("long").cast("string")
    )

    # Trim organism name
    .withColumn(
        "org_name",
        F.when(
            F.trim(F.col("org_name")) == "",
            F.lit(None)
        ).otherwise(
            F.trim(F.col("org_name"))
        )
    )

    # Convert antibiotic ID to string and preserve nulls
    .withColumn(
        "ab_itemid",
        F.col("ab_itemid").cast("long").cast("string")
    )

    # Trim antibiotic name
    .withColumn(
        "ab_name",
        F.when(
            F.trim(F.col("ab_name")) == "",
            F.lit(None)
        ).otherwise(
            F.trim(F.col("ab_name"))
        )
    )

    # Trim dilution text
    .withColumn(
        "dilution_text",
        F.when(
            F.trim(F.col("dilution_text")) == "",
            F.lit(None)
        ).otherwise(
            F.trim(F.col("dilution_text"))
        )
    )

    # Trim dilution comparison
    .withColumn(
        "dilution_comparison",
        F.when(
            F.trim(F.col("dilution_comparison")) == "",
            F.lit(None)
        ).otherwise(
            F.trim(F.col("dilution_comparison"))
        )
    )

    # Convert dilution value to DoubleType
    .withColumn(
        "dilution_value",
        F.col("dilution_value").cast("double")
    )

    # Trim and uppercase susceptibility interpretation
    .withColumn(
        "interpretation",
        F.when(
            F.trim(F.col("interpretation")) == "",
            F.lit(None)
        ).otherwise(
            F.upper(F.trim(F.col("interpretation")))
        )
    )

    # Trim comments and preserve nulls
    .withColumn(
        "comments",
        F.when(
            F.trim(F.col("comments")) == "",
            F.lit(None)
        ).otherwise(
            F.trim(F.col("comments"))
        )
    )

    # Remove duplicate microbiology event IDs
    .dropDuplicates(["microevent_id"])
)

# Check that every organism ID maps to one organism name
organism_mapping_issues = (
    microbiology_clean
    .filter(F.col("org_itemid").isNotNull())
    .groupBy("org_itemid")
    .agg(
        F.countDistinct("org_name").alias("organism_name_count"),
        F.sum(
            F.when(
                F.col("org_name").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_name_count")
    )
    .filter(
        (F.col("organism_name_count") != 1)
        | (F.col("missing_name_count") > 0)
    )
)

# Check that every antibiotic ID maps to one antibiotic name
antibiotic_mapping_issues = (
    microbiology_clean
    .filter(F.col("ab_itemid").isNotNull())
    .groupBy("ab_itemid")
    .agg(
        F.countDistinct("ab_name").alias("antibiotic_name_count"),
        F.sum(
            F.when(
                F.col("ab_name").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_name_count")
    )
    .filter(
        (F.col("antibiotic_name_count") != 1)
        | (F.col("missing_name_count") > 0)
    )
)

# Check for invalid interpretation codes
interpretation_issues = (
    microbiology_clean
    .filter(
        F.col("interpretation").isNotNull()
        & ~F.col("interpretation").isin("S", "R", "I")
    )
    .select("interpretation")
    .distinct()
)

# Stop dimension creation if mappings are inconsistent
assert organism_mapping_issues.limit(1).count() == 0, (
    "An org_itemid maps to multiple or missing organism names."
)

assert antibiotic_mapping_issues.limit(1).count() == 0, (
    "An ab_itemid maps to multiple or missing antibiotic names."
)

assert interpretation_issues.limit(1).count() == 0, (
    "Unexpected susceptibility interpretation codes were found."
)

# Create the organism dimension
dim_organism = (
    microbiology_clean
    .select(
        "org_itemid",
        "org_name"
    )
    .filter(
        F.col("org_itemid").isNotNull()
        & F.col("org_name").isNotNull()
    )
    .dropDuplicates(["org_itemid"])
)

# Create the antibiotic dimension
dim_antibiotic = (
    microbiology_clean
    .select(
        "ab_itemid",
        "ab_name"
    )
    .filter(
        F.col("ab_itemid").isNotNull()
        & F.col("ab_name").isNotNull()
    )
    .dropDuplicates(["ab_itemid"])
)

# Create the susceptibility interpretation dimension
dim_susceptibility_interpretation = (
    microbiology_clean
    .select("interpretation")
    .filter(
        F.col("interpretation").isin("S", "R", "I")
    )
    .dropDuplicates(["interpretation"])
    .withColumn(
        "interpretation_description",
        F.when(
            F.col("interpretation") == "S",
            F.lit("Susceptible")
        )
        .when(
            F.col("interpretation") == "R",
            F.lit("Resistant")
        )
        .when(
            F.col("interpretation") == "I",
            F.lit("Intermediate")
        )
    )
)

# Create the fact table with dimension IDs only
fact_microbiologyevents = (
    microbiology_clean
    .drop(
        "org_name",
        "ab_name"
    )
)

# Store the fact and dimension DataFrames
silver_dfs["microbiologyevents"] = fact_microbiologyevents
silver_dfs["dim_organism"] = dim_organism
silver_dfs["dim_antibiotic"] = dim_antibiotic
silver_dfs[
    "dim_susceptibility_interpretation"
] = dim_susceptibility_interpretation